# SETUP CCEK

In [ ]:
# Cek library yang tersedia
import numpy as np
import pandas as pd
import sys
import platform
import os

In [ ]:


print("Python version:", sys.version)
print("Platform:", platform.platform())



print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
NumPy: 2.4.6
Pandas: 2.2.3


#CLONE REPO

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive/BAJANG TACTIC"

repo_path = os.path.join(
    base_path,
    "official-repository",
    "more-than-a-hack-2027"
)

os.makedirs(os.path.dirname(repo_path), exist_ok=True)

print("Lokasi repository:", repo_path)

Lokasi repository: /content/drive/MyDrive/BAJANG TACTIC/official-repository/more-than-a-hack-2027


In [ ]:
if not os.path.exists(repo_path):
    !git clone https://github.com/metrica-sports/more-than-a-hack-2027.git "{repo_path}"
    print("Repository berhasil di-clone!")
else:
    print("Repository sudah tersedia.")

Repository sudah tersedia.


In [ ]:
os.chdir(repo_path)

print("Daftar file repository:")
!ls

print("\nDaftar file starter:")
!ls starter

Daftar file repository:
CHALLENGE.md  evaluation     LICENSE		       requirements.txt
CITATION.cff  EVALUATION.md  OVERVIEW.md	       RULES.md
data	      FAQ.md	     README.md		       starter
DATA.md       games.py	     requirements-starter.txt  SUBMISSION.md

Daftar file starter:
01_smart_tagging.ipynb	   baseline.py	       load_smart_tagging.py
02_tracking_atd.ipynb	   load_atd.py	       make_dummy_submission.py
03_first_submission.ipynb  load_homography.py  utils.py


#baca dokumentasi resmi

In [ ]:
repo_path = "/content/drive/MyDrive/BAJANG TACTIC/official-repository/more-than-a-hack-2027"

readme_path = os.path.join(repo_path, "README.md")

with open(readme_path, "r", encoding="utf-8") as file:
    readme_content = file.read()

print(readme_content)

# More than a Hack 2027 — Phase 1: Ball Possession

**Organised by [Metrica Sports](https://metrica-sports.com), [FC Barcelona](https://www.fcbarcelona.com)
and the [Mobile World Capital Barcelona](https://www.mwcbarcelona.com)**, at the Talent Arena in Barcelona.

A football data challenge on real professional match data, open to analysts, data scientists and
engineers. You are given one complete LaLiga match — two camera feeds, automatic tracking of every
player and the ball, automatic tactical tagging, and the camera-to-pitch geometry — and one
question to answer with it.

---

## The task

**For every frame of the match, say who has the ball.**

```
frame,label
11185,A        away in possession, ball alive
11186,A
...
18240,D        dead ball
...
18684,H        home in possession, ball alive
```

That is the whole task. One CSV, one row per video frame, 199,319 rows. It is scored automatically
against a private frame-by-frame ground truth, so there is a number at the end and no arg

In [ ]:
data_path = os.path.join(repo_path, "DATA.md")

with open(data_path, "r", encoding="utf-8") as file:
    data_content = file.read()

print(data_content)

# The data

One match: **FC Barcelona vs Athletic Club**, LaLiga EA Sports 2024-25, matchday 2.
Home is Barcelona (`H`), away is Athletic (`A`).

It was recorded twice, by two different cameras, and you get tracking for both.

| | **tactical** | **broadcast** |
|---|---|---|
| **role** | **the reference. Row *i* of your submission is frame *i* of this video** | optional extra input, **not synchronised** to tactical |
| video | 199,319 frames, 25 fps, 1920×1080 | its own timeline, with cuts and replays |
| tracking (ATD) | 199,319 records | 164,488 records |
| homography | 199,319 rows, **100%** usable | 164,488 rows, **65.8%** usable |
| Smart Tagging | 638 instances | — |
| track slots | 46 | 51 |
| ball visible | 56.9% of frames | 50.5% of frames |

Everything is fair game, in any combination, as input to an algorithm. A system built only on the
ball track is as legitimate as one that reads the video. There is no intended solution. What is not
allowed is labelling or correcting any o

# mengenali isi folder data

In [ ]:
# Memeriksa isi folder data
data_path = os.path.join(repo_path, "data")

print("Isi folder data:")
for root, dirs, files in os.walk(data_path):
    print("Folder:", root)
    print("Subfolder:", dirs)
    print("File:", files)

Isi folder data:
Folder: /content/drive/MyDrive/BAJANG TACTIC/official-repository/more-than-a-hack-2027/data
Subfolder: []
File: ['README.md']


In [ ]:
# Memeriksa file-file starter
starter_path = os.path.join(repo_path, "starter")

print("Daftar file starter:\n")

for file in os.listdir(starter_path):
    print(file)

Daftar file starter:

01_smart_tagging.ipynb
02_tracking_atd.ipynb
03_first_submission.ipynb
baseline.py
load_atd.py
load_homography.py
load_smart_tagging.py
make_dummy_submission.py
utils.py


# memahami load_atd.py

In [ ]:
atd_file = os.path.join(
    repo_path,
    "starter",
    "load_atd.py"
)

# Membaca kode sumber
with open(atd_file, "r", encoding="utf-8") as file:
    atd_code = file.read()

print(atd_code)

"""Load the provided Automatic Tracking Data and line it up with the video frames you submit.

The ATD is Metrica's FIFA/EPTS export -- a metadata XML plus a raw text file:

    data/tactical/BAR-ATH_tactical_atd.xml
    data/tactical/BAR-ATH_tactical_atd.txt

games.asset_paths() builds both, so nothing here spells a filename out.

The match was recorded twice and each recording has its own ATD. This module defaults to the
tactical one, because that is the recording a submission is indexed to. The broadcast ATD is a
separate 164488-frame dataset on its own timeline:

    load(source="broadcast")

kloppy reads that pair directly, so this module is thin on purpose. What it adds is the part that
is easy to get wrong and expensive to notice: the join between tracking frames and video frames.

    pip install -r requirements-starter.txt
    python starter/load_atd.py                  # prints a summary and a few aligned frames

What you actually need to know
------------------------------
T

# menguji pemetaan frame

In [ ]:
# Simulasi fungsi pemetaan frame dari load_atd.py

def video_frame_of(frame_id):
    return int(frame_id) - 1

def frame_id_of(video_frame):
    return int(video_frame) + 1


# Contoh nomor frame ATD
atd_frames = [1, 2, 3, 100, 199319]

print("Pengujian Frame Alignment")
print("-" * 40)

for frame_id in atd_frames:
    video_frame = video_frame_of(frame_id)

    print(
        "ATD frame:", frame_id,
        "| Video frame:", video_frame,
        "| Kembali:", frame_id_of(video_frame)
    )

Pengujian Frame Alignment
----------------------------------------
ATD frame: 1 | Video frame: 0 | Kembali: 1
ATD frame: 2 | Video frame: 1 | Kembali: 2
ATD frame: 3 | Video frame: 2 | Kembali: 3
ATD frame: 100 | Video frame: 99 | Kembali: 100
ATD frame: 199319 | Video frame: 199318 | Kembali: 199319


# mempelajari Smart Tagging

In [ ]:
# Membaca source code load_smart_tagging.py
smart_tagging_file = os.path.join(
    repo_path,
    "starter",
    "load_smart_tagging.py"
)

with open(smart_tagging_file, "r", encoding="utf-8") as file:
    smart_tagging_code = file.read()

print(smart_tagging_code)

"""Read the Smart Tagging XML and convert its intervals to video frames.

Smart Tagging is Metrica's automatic tactical annotation of the tactical video: phases of play and
set pieces, each an interval with a code and a set of labels. It is a SportsCode XML file, so this
needs nothing but the standard library.

    python starter/load_smart_tagging.py

What it is good for: every phase carries a Team tag, and every set piece is a restart at the end
of a dead-ball stretch. That is a free prior on both halves of the label you are producing.

What it is not: ground truth. It is inferred automatically and contains false positives and false
negatives, and its intervals are *phases*, not possessions -- a Build Up can span one possession,
stop short of one, or cover two. Copy an interval across as a label and you inherit its errors.

TIME BASE, verified on the delivered file: <start> and <end> are seconds from the first frame of
the tactical video, so seconds x 25 is a video frame and nothing 

# memahami XML menggunakan data sintetis

In [ ]:
import xml.etree.ElementTree as ET

# Contoh XML sintetis
xml_data = """
<annotations>
    <instance>
        <start>120.0</start>
        <end>135.0</end>
        <code>BUILD UP</code>

        <label>
            <group>Team</group>
            <text>Barcelona</text>
        </label>

        <label>
            <group>Half</group>
            <text>1st Half</text>
        </label>
    </instance>
</annotations>
"""

# Membaca XML
root = ET.fromstring(xml_data)

# Mengambil instance
for instance in root.iter("instance"):

    start = float(instance.findtext("start"))
    end = float(instance.findtext("end"))
    code = instance.findtext("code")

    tags = {
        label.findtext("group"): label.findtext("text")
        for label in instance.findall("label")
    }

    print("Code:", code)
    print("Start:", start)
    print("End:", end)
    print("Tags:", tags)

Code: BUILD UP
Start: 120.0
End: 135.0
Tags: {'Team': 'Barcelona', 'Half': '1st Half'}


In [ ]:
homography_file = os.path.join(
    repo_path,
    "starter",
    "load_homography.py"
)

with open(homography_file, "r", encoding="utf-8") as file:
    homography_code = file.read()

print(homography_code)

"""Read the per-frame homography and move points between the pitch and the image.

The homography is what connects the two things you are given: it maps a position on the pitch to
the pixel that shows it, and back. That makes it the tool for looking at your own mistakes -- take
a spurious transition out of reports/<name>_transitions.csv, project the players onto that frame
of the video, and see what fooled the model.

    pip install -r requirements.txt          # numpy and pandas, nothing else
    python starter/load_homography.py        # summary, both sources

    data/tactical/BAR-ATH_tactical_homography.csv
    data/broadcast/BAR-ATH_broadcast_homography.csv

One row per video frame of that recording, `frame` counting from 0. For the tactical file that is
the same index as a submission row, so row i is the homography for the frame you label in row i.

    frame,h00,h01,h02,h10,h11,h12,h20,h21,h22,reprojection_error,valid

Columns 1..9 are the 3x3 matrix in row-major order, which i

# data sintetis Homography

In [ ]:
def project(matrix, x, y):
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)

    denominator = (
        matrix[2, 0] * x +
        matrix[2, 1] * y +
        matrix[2, 2]
    )

    return (
        (matrix[0, 0] * x + matrix[0, 1] * y + matrix[0, 2]) / denominator,
        (matrix[1, 0] * x + matrix[1, 1] * y + matrix[1, 2]) / denominator
    )

In [ ]:
# Membuat matriks identitas 3x3
H = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=float)

# Koordinat pemain pada lapangan
x = 0.5
y = 0.7

# Melakukan proyeksi
pixel_x, pixel_y = project(H, x, y)

print("Koordinat lapangan:", (x, y))
print("Koordinat hasil proyeksi:", (pixel_x, pixel_y))

Koordinat lapangan: (0.5, 0.7)
Koordinat hasil proyeksi: (np.float64(0.5), np.float64(0.7))


# belajar Baseline.py

In [ ]:
# Membaca source code baseline.py

baseline_file = os.path.join(
    repo_path,
    "starter",
    "baseline.py"
)

with open(baseline_file, "r", encoding="utf-8") as file:
    baseline_code = file.read()

print(baseline_code)

"""The reference baseline: whoever is closest to the ball has it.

This is the floor to beat, and it is deliberately the simplest thing that is not a constant guess.
Two steps, no model, no training:

  1. For every frame where the tracking sees the ball, find the nearest tracked player and give
     that player's team the ball. Where the ball is not visible, say D.
  2. Smooth the result with a centred majority vote, because step 1 alone flickers.

    pip install -r requirements.txt          # numpy and pandas, nothing else
    python starter/baseline.py               # writes four submissions and scores them

numpy only, on purpose: it runs in the same environment as the scorer, so getting to a scored
submission needs no kloppy, no video decoder and no plotting stack. Reading the raw ATD by hand
also shows the file format more plainly than a loader would. Expect it to take a minute or two --
it streams 199319 lines of tracking.

Why step 2 is the whole point
------------------------

In [1]:
import numpy as np

# Posisi bola
ball = np.array([0.50, 0.50])

# Posisi pemain
players = np.array([
    [0.52, 0.51],  # Pemain 1
    [0.30, 0.40],  # Pemain 2
    [0.60, 0.55],  # Pemain 3
    [0.80, 0.70]   # Pemain 4
])

# Tim setiap pemain
teams = np.array([
    "Home",
    "Home",
    "Away",
    "Away"
])

print("Posisi bola:", ball)
print("\nPosisi pemain:")
print(players)

print("\nTim pemain:")
print(teams)

Posisi bola: [0.5 0.5]

Posisi pemain:
[[0.52 0.51]
 [0.3  0.4 ]
 [0.6  0.55]
 [0.8  0.7 ]]

Tim pemain:
['Home' 'Home' 'Away' 'Away']
